# Rare-term weighting on Kaggle — baseline vs. hapax_binary vs. inverse_sqrt_freq

Adapted from `scripts/kaggle_deberta_grid.ipynb` (same environment pattern: data cloned via
git, not uploaded; pinned installs; GPU T4 x2). This notebook does **not** touch
`origin/main` — the rare-term-weighting change lives only on the local
`feature/rare-term-weighting` branch and is never pushed. It is applied here as a patch,
written into a fresh clone of `origin/main`, so nothing about this run depends on that
branch existing anywhere but on Khaled's own machine.

**Two things to set in the right-hand sidebar before running:**
1. **Session options → Accelerator → GPU T4 x2** (or P100)
2. **Session options → Internet → On** (needs a phone-verified account) — the clone and the
   HuggingFace download both need it.

Model/LR/epochs default to **deberta-v3-base, 1e-5, 5 epochs — T10's selected config**
(see cell 6). Swap to `bert-base-cased` / `3e-5` there if a faster, non-final run is what's
wanted instead.

In [1]:
# 1. Kaggle guard, internet, GPU, and the command helper.
import os, sys, socket, subprocess, pathlib

if not pathlib.Path('/kaggle').is_dir():
    raise SystemExit('This notebook is for Kaggle. On Colab use colab_deberta_grid.ipynb.')

try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError as e:
    raise SystemExit(
        f'No internet ({e}). Kaggle disables it by default.\n'
        'Sidebar -> Session options -> Internet -> On (needs a phone-verified account).')

import torch
assert torch.cuda.is_available(), 'No GPU. Sidebar -> Session options -> Accelerator -> GPU.'
print(torch.cuda.get_device_name(0), '|', torch.__version__, '| cuda', torch.version.cuda)

WORK = pathlib.Path('/kaggle/working')        # persisted as notebook Output
REPO = pathlib.Path('/tmp/ate-acter')         # scratch: repo + corpus stay out of Output

def run(*args, cwd=None):
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         cwd=cwd, env=env)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {" ".join(str(a) for a in args)}')


Tesla T4 | 2.10.0+cu128 | cuda 12.8


In [2]:
# 2. Clone the project (Specific Branch) and the corpus into /tmp.
import shutil

BRANCH_NAME = "feature/rare-term-weighting"  

shutil.rmtree(REPO, ignore_errors=True)

run('git', 'clone', '-b', BRANCH_NAME, '-q', 'https://github.com/ahmedwaleedaref/ATE-ACTER.git', REPO)

run('git', 'clone', '-q', 'https://github.com/AylaRT/ACTER.git', REPO / 'data/raw/ACTER')
run('git', 'checkout', '-q', 'f05b09e985cad37eeaa8daa8b3f383197aa5324e',
    cwd=REPO / 'data/raw/ACTER')

assert (REPO / 'data/raw/ACTER/en/htfl/annotated').is_dir(), 'ACTER checkout looks wrong'

print(subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                    capture_output=True, text=True).stdout)

21c927e edit on test_schedule



In [3]:
# 4. Pinned installs. torch/numpy are Kaggle's -- forcing them breaks its CUDA
#    build, exactly as on Colab.
run(sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==5.16.1', 'tokenizers==0.23.1', 'safetensors==0.8.0',
    'huggingface_hub==1.29.0', 'sentencepiece==0.2.2', 'protobuf==7.36.0',
    'PyYAML==6.0.3', 'pytest==8.3.2')

def _pip(*a):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a],
                          capture_output=True, text=True).returncode == 0
HAVE_SEQEVAL = (_pip('--no-build-isolation', 'seqeval==1.2.2')
                or (_pip('setuptools<81', 'wheel')
                    and _pip('--no-build-isolation', 'seqeval==1.2.2')))
print('seqeval:', 'installed' if HAVE_SEQEVAL else 'UNAVAILABLE -- training unaffected')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 340.4/340.4 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.8/341.8 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.2

In [4]:
# 5. Run the full test suite -- including tests/test_rare_terms.py, which now has
#    torch available, so the weighted-loss-math regression test actually runs here
#    (it only skips locally, where torch is missing).
skip = [] if HAVE_SEQEVAL else ['--ignore=tests/test_seqeval_agreement.py']
run(sys.executable, '-m', 'pytest', 'tests/', '-v', *skip, cwd=REPO)


============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.3.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /tmp/ate-acter
configfile: pyproject.toml
plugins: anyio-4.13.0, typeguard-4.5.1, langsmith-0.7.34
collecting ... collected 56 items

tests/test_alignment_gate.py::test_alignment_gate[corp] PASSED           [  1%]
tests/test_alignment_gate.py::test_alignment_gate[equi] PASSED           [  3%]
tests/test_alignment_gate.py::test_alignment_gate[wind] PASSED           [  5%]
tests/test_alignment_gate.py::test_alignment_gate[htfl] PASSED           [  7%]
tests/test_alignment_gate.py::test_align_labels_on_a_hand_checked_sentence PASSED [  8%]
tests/test_alignment_gate.py::test_collator_pads_labels_with_ignore_index PASSED [ 10%]
tests/test_alignment_gate.py::test_short_sentence_filter_is_explicit_and_wind_only PASSED [ 12%]
tests/test_alignment_gate.py::test_sent_idx_survives_the_filter PASSED   [ 14

In [5]:
# 6. The three conditions: baseline (weighting off), hapax_binary, inverse_sqrt_freq.
#    5 seeds each -- 15 runs total, matching the project's own seed convention (T8/T9).
#    Model/LR/epochs = T10's selected config. Swap to bert-base-cased/3e-5 here for a
#    faster, non-final run instead.
import json

MODEL, LR, EPOCHS = 'microsoft/deberta-v3-base', '1e-5', 5
SEEDS = (42, 43, 44, 45, 46)
CONDITIONS = [
    ('baseline', []),
    ('hapax_binary', ['--rare-term-weighting', '--rare-term-formula', 'hapax_binary']),
    ('inverse_sqrt_freq', ['--rare-term-weighting', '--rare-term-formula', 'inverse_sqrt_freq']),
]

for name, flags in CONDITIONS:
    group = f'rare_term_weighting/{name}'
    src = REPO / 'results/runs' / group
    print(f'\n########## {name} ##########')
    for s in SEEDS:
        run(sys.executable, '-m', 'src.models.run_train',
            '--model', MODEL, '--lr', LR, '--epochs', EPOCHS,
            '--group', group, '--seed', s,
            '--reason', f'rare-term weighting: {name}', *flags, cwd=REPO)
        r = json.loads((src / f'seed_{s}.json').read_text())
        print(f"    SEED {s}: best_epoch={r['best_epoch']} equi={r['best_equi_f1']:.4f} "
              f"htfl={r['htfl_f1']:.4f} collapsed={r['collapsed']} {r['wall_time_sec']}s")
    dest = WORK / group
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.rmtree(dest, ignore_errors=True)
    shutil.copytree(src, dest)
    print(f'  -> copied to {dest} (persists as notebook Output; survives a dropped session)')



########## baseline ##########
overrides: model_name=microsoft/deberta-v3-base, num_epochs=5, learning_rate=1e-05
run 20260916-155952_microsoft-deberta-v3-base_lr1e-05_e5_seed42
device cuda (Tesla T4)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 723.08it/s]
[transformers] DebertaV2ForTokenClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head

In [6]:
# 7. What to download: everything under /kaggle/working/rare_term_weighting.
#    Drop these folders into results/runs/rare_term_weighting/ locally (on the
#    feature/rare-term-weighting branch), then aggregate/compare the 3 conditions.
#    T11 (hapax-vs-rest breakdown on htfl) still needs to be built before these numbers
#    answer "did it work on hapax specifically" -- this cell only gets the runs done.
for p in sorted((WORK / 'rare_term_weighting').rglob('seed_*.json')):
    print(' ', p.relative_to(WORK), f'{p.stat().st_size/1024:.0f} KB')


  rare_term_weighting/baseline/seed_42.json 7 KB
  rare_term_weighting/baseline/seed_43.json 7 KB
  rare_term_weighting/baseline/seed_44.json 7 KB
  rare_term_weighting/baseline/seed_45.json 7 KB
  rare_term_weighting/baseline/seed_46.json 7 KB
  rare_term_weighting/hapax_binary/seed_42.json 7 KB
  rare_term_weighting/hapax_binary/seed_43.json 7 KB
  rare_term_weighting/hapax_binary/seed_44.json 7 KB
  rare_term_weighting/hapax_binary/seed_45.json 8 KB
  rare_term_weighting/hapax_binary/seed_46.json 7 KB
  rare_term_weighting/inverse_sqrt_freq/seed_42.json 8 KB
  rare_term_weighting/inverse_sqrt_freq/seed_43.json 8 KB
  rare_term_weighting/inverse_sqrt_freq/seed_44.json 8 KB
  rare_term_weighting/inverse_sqrt_freq/seed_45.json 7 KB
  rare_term_weighting/inverse_sqrt_freq/seed_46.json 8 KB
